# Forecast, model, explain: three bank questions answered with public data

Order: **forecast → target → credit → ATM network → AI Act → recommendations**.

Public data: ECB (Belgium), UCI Bank Marketing and Default of Credit Card Clients (CC BY 4.0; Portugal 2008-2010 and Taiwan 2005, not Belgian clients), Regulation (EU) 2024/1689.
Simulated data: 60 ATMs of a fictitious "Bank B". Run the scripts listed in the README first; this notebook reads their outputs and re-runs one backtest step by step.

In [1]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
P = ROOT / "data" / "processed"
F = json.loads((P / "findings.json").read_text(encoding="utf-8"))
pd.set_option("display.precision", 3)

## 1. Forecast Belgian mortgage volumes (PUBLIC DATA, ECB)

In [2]:
e = pd.read_csv(P / "ecb_series.csv")
e.groupby(["series", "key"]).agg(first=("month", "min"), last=("month", "max"), n=("value", "size"), unit_mult=("UNIT_MULT", "first"))

,,first,last,n,unit_mult
series,key,,,,
deposits_hh_overnight,BSI.M.BE.N.A.L21.A.1.U2.2250.Z01.E,2015-01,2026-09,141,6
mortgage_rate_new,MIR.M.BE.B.A2C.A.R.A.2250.EUR.N,2015-01,2026-09,141,0
mortgage_volume_new,MIR.M.BE.B.A2C.A.B.A.2250.EUR.N,2015-01,2026-09,141,6


In [3]:
pd.read_csv(P / "fc_metrics.csv")   # 12 rolling origins, horizons 1-12 months

,model,mape,mase,cov80,cov95
0,ETS,0.084,0.314,0.965,0.993
1,SARIMAX + rate,0.110,0.415,0.986,0.993
2,Seasonal naive,0.150,0.559,NaN,NaN


In [4]:
pd.read_csv(P / "fc_scenarios.csv").groupby("scenario").pred.sum().div(1000).round(1).rename("next 12 months, EUR bn")

scenario
+50 bp         33.8
-50 bp         51.7
Stable rate    41.8
Name: next 12 months, EUR bn, dtype: float64

In [5]:
d = F["forecast"]["deposits"]
{k: d[k] for k in ("break_month", "growth_before_pct_yr", "growth_after_pct_yr", "peak_month", "peak_value_m", "last_value_m")}

{'break_month': '2021-10',
 'growth_before_pct_yr': 10.425782623577895,
 'growth_after_pct_yr': -3.9098226993059204,
 'peak_month': '2022-06',
 'peak_value_m': 136122.0,
 'last_value_m': 108680.0}

### Re-run the ETS backtest yourself, step by step

Change `H` (horizon in months) and run the cell. Each step is commented.

In [6]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

H = 6                                                     # <- horizon to test: 1 to 12 months ahead
w = e.pivot(index="month", columns="series", values="value")
y = w["mortgage_volume_new"]                              # new mortgage volume, EUR millions
y.index = pd.PeriodIndex(y.index, freq="M")
origins = pd.period_range("2024-10", "2025-09", freq="M")  # 12 forecast origins
errors = []
for o in origins:
    train = y[:o]                                         # 1. only data known at the origin
    model = ETSModel(np.log(train), error="add", trend="add", damped_trend=True,
                     seasonal="add", seasonal_periods=12).fit(disp=False)   # 2. fit on log volume
    pred = np.exp(model.forecast(H).iloc[-1])             # 3. forecast H months ahead, back to EUR
    actual = y[o + H]                                     # 4. what really happened
    errors.append(abs(actual - pred) / actual)            # 5. absolute % error
print(f"ETS, horizon {H} months: MAPE {np.mean(errors):.1%} over {len(errors)} origins")

ETS, horizon 6 months: MAPE 7.5% over 12 origins


## 2. Who to target (PUBLIC DATA, UCI Bank Marketing)

In [7]:
pd.read_csv(P / "bm_model_comparison.csv")   # 'with duration' = information leak, not used

,model,features,auc
0,Logistic,without duration,0.801
1,Gradient boosting,without duration,0.817
2,Logistic,with duration (leak),0.942
3,Gradient boosting,with duration (leak),0.955


In [8]:
m = F["marketing"]
{k: round(m[k], 3) for k in ("gain_top10", "gain_top20", "gain_top30", "lift_top20", "auc_oot_logistic", "auc_oot_gb")}

{'gain_top10': 0.481,
 'gain_top20': 0.664,
 'gain_top30': 0.751,
 'lift_top20': 3.319,
 'auc_oot_logistic': 0.749,
 'auc_oot_gb': 0.67}

In [9]:
b = pd.read_csv(P / "bm_segments.csv")
b[b.segment.isin(["previous campaign outcome", "contact channel", "calls in this campaign"])]

,segment,level,calls,subscription_rate
3,calls in this campaign,1,17642,0.130
4,calls in this campaign,2,10570,0.115
5,calls in this campaign,3,5341,0.107
6,calls in this campaign,4,2651,0.094
7,calls in this campaign,5+,4984,0.061
8,contact channel,cellular,26144,0.147
9,contact channel,telephone,15044,0.052
25,previous campaign outcome,failure,4252,0.142
26,previous campaign outcome,nonexistent,35563,0.088
27,previous campaign outcome,success,1373,0.651


In [10]:
pd.read_csv(P / "bm_fairness_age.csv")   # selection rate at a 20% call budget, aggregates only

,age_band,n,subscription_rate,selected_rate
0,<30,1095,0.153,0.307
1,30-39,3452,0.104,0.171
2,40-49,2096,0.077,0.140
3,50-59,1351,0.106,0.167
4,60+,244,0.393,0.828


## 3. Credit default (PUBLIC DATA, UCI Default of Credit Card Clients)

In [11]:
pd.read_csv(P / "credit_model_comparison.csv")

,model,auc,gini,ks,brier
0,Logistic scorecard,0.754,0.509,0.399,0.139
1,Gradient boosting,0.775,0.551,0.420,0.136


In [12]:
c = F["credit"]
{k: round(c[k], 3) for k in ("cal_max_gap", "threshold_best", "threshold_theory", "cost_best", "cost_at_050", "acceptance_at_best", "psi_aug_sep", "psi_stress")}

{'cal_max_gap': 0.02,
 'threshold_best': 0.15,
 'threshold_theory': 0.167,
 'cost_best': 0.593,
 'cost_at_050': 0.767,
 'acceptance_at_best': 0.605,
 'psi_aug_sep': 0.04,
 'psi_stress': 0.172}

In [13]:
pd.read_csv(P / "credit_fairness.csv")   # sex and age are not model inputs; every group n >= 5

,attribute,grp,n,observed_default_rate,mean_predicted_pd,acceptance_rate,good_declined_rate,bad_accepted_rate
0,age band,30-39,3332,0.198,0.204,0.642,0.283,0.340
1,age band,40-49,1958,0.239,0.216,0.617,0.296,0.342
2,age band,50+,828,0.260,0.232,0.579,0.333,0.326
3,age band,<30,2882,0.224,0.237,0.560,0.355,0.267
4,sex,female,5437,0.210,0.216,0.614,0.309,0.323
5,sex,male,3563,0.239,0.225,0.590,0.320,0.305


In [14]:
pd.read_csv(P / "credit_importance.csv")

,variable,auc_drop
0,months_late_5m,0.083
1,status_last,0.041
2,LIMIT_BAL,0.016
3,utilisation,0.007
4,bill_trend,0.003
5,paid_ratio,0.001


## 4. ATM network: data quality, forecast, drift (SIMULATED)

In [15]:
pd.read_csv(P / "dq_report.csv")[["check", "status", "rows_affected", "owner"]]

,check,status,rows_affected,owner
0,"unique (date, atm_id)",FAIL,9,data engineering
1,zero vs missing,FAIL,36,data engineering
2,required fields not null,PASS,0,data engineering
3,freshness (< 08:00 next day),WARN,2,data engineering
4,rows per day = active ATMs,FAIL,1,data engineering
5,no rows after decommissioning,PASS,0,data science
6,extreme values explained,WARN,37,business + data science
7,network-level drop vs last week,PASS,0,data science


In [16]:
a = F["atm"]
{k: (round(a[k], 3) if isinstance(a[k], float) else a[k]) for k in ("mape_model", "mape_naive", "mape_v1_random_lag1", "mape_random_split", "drift_detect_mall", "drift_days_to_detect", "simpson_holds")}

{'mape_model': 0.138,
 'mape_naive': 0.215,
 'mape_v1_random_lag1': 0.113,
 'mape_random_split': 0.134,
 'drift_detect_mall': '2026-09-02',
 'drift_days_to_detect': 2,
 'simpson_holds': True}

In [17]:
pd.read_csv(P / "simpson.csv")

,region,site_type,n,mape,share_of_region
0,Brussels,ALL,3454,0.177,1.000
1,Flanders,ALL,6414,0.125,1.000
2,Brussels,city_centre,710,0.101,0.206
3,Flanders,city_centre,1902,0.112,0.297
4,Brussels,mall,474,0.115,0.137
5,Flanders,mall,1426,0.127,0.222
6,Brussels,rural,238,0.084,0.069
7,Flanders,rural,2374,0.100,0.370
8,Brussels,station,2032,0.228,0.588
9,Flanders,station,712,0.241,0.111


In [18]:
pd.read_csv(P / "atm_refill.csv").groupby("policy")[["visits", "stockout_hours", "cost_eur"]].sum().round(0)

,visits,stockout_hours,cost_eur
policy,,,
fixed,1035,0.0,160656.0
forecast,749,23.0,117849.0


## 5. EU AI Act (Regulation (EU) 2024/1689): citations used. Not legal advice.

In [19]:
pd.read_csv(P / "ai_act_citations.csv").query("ref.str.contains('Annex|Recital|113|Article 1[0-5]|Article 9')", engine="python")

,ref,title
1,Article 9,Risk management system
2,Article 10,Data and data governance
3,Article 11,Technical documentation
4,Article 12,Record-keeping
5,Article 13,Transparency and provision of information to d...
6,Article 14,Human oversight
7,Article 15,"Accuracy, robustness and cybersecurity"
15,"Annex III, point 5(b)",(b) AI systems intended to be used to evaluate...
16,Recital 58,AI systems used to evaluate the credit score o...
17,Article 113 (application),It shall apply from 2 August 2026.


## 6. Recommendations

1. Plan mortgage volumes on three rate scenarios, as ranges.
2. Score clients and call the top 20% first; never score on call duration; retrain before each campaign.
3. Use the explainable scorecard with a cost-based cut-off; a person reviews each decline; review the under-30 gap.
4. Data contract with the data engineers; two-level drift monitor; forecast-driven ATM refills.
5. Treat credit scoring as high-risk: model card, logs, human oversight, impact assessment before go-live.